In [1]:
# Imports
import os
import numpy as np
import nibabel as nib
from scipy.ndimage import binary_dilation
from scipy.ndimage import zoom
import shutil
import pydicom
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
from rt_utils import RTStructBuilder
import re
from scipy import ndimage

In [ ]:
# 1. Extract modalities and file locations

def classify_dicom_folder(root_dir, output_excel):
    results = []

    # Walk through each subfolder
    for subdir, _, files in os.walk(root_dir):
        if not files:
            continue  # skip empty folders
        
        # Take the first file in this folder
        first_file = os.path.join(subdir, files[0])
        
        try:
            ds = pydicom.dcmread(first_file, stop_before_pixels=True)
            modality = ds.Modality if "Modality" in ds else "Unknown"
        except Exception as e:
            modality = f"Error: {str(e)}"

        # Extract ID from folder name (last part of path)
        folder_id = os.path.basename(subdir)

        results.append({
            "ID": folder_id,
            "Folder": subdir,
            "Modality": modality,
            "Number_of_Files": len(files)
        })

    # Save to Excel
    df = pd.DataFrame(results)
    df.to_excel(output_excel, index=False)
    print(f"Excel file saved: {output_excel}")

# Usage of function
classify_dicom_folder(r"C:\Users\VetteSPM\OneDrive - UMCG\Documents\TCIA_Lung", r"C:\Users\VetteSPM\OneDrive - UMCG\Documents\dicom_summary.xlsx")


In [ ]:
# 2. Extract available ROIs from the RTSTRUCT file

def extract_rois_from_rtstruct(input_excel, output_excel):
    # Load your previous Excel
    df = pd.read_excel(input_excel)

    roi_results = []

    for _, row in df.iterrows():
        if str(row["Modality"]).upper() != "RTSTRUCT":
            continue  # skip non-RTSTRUCT

        folder = row["Folder"]
        if not os.path.exists(folder):
            continue

        # Pick first file in RTSTRUCT folder
        files = [f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))]
        if not files:
            continue

        first_file = os.path.join(folder, files[0])

        try:
            ds = pydicom.dcmread(first_file, stop_before_pixels=True)
            if hasattr(ds, "StructureSetROISequence"):
                for roi in ds.StructureSetROISequence:
                    roi_name = roi.ROIName if "ROIName" in roi else "Unnamed"
                    roi_results.append({
                        "ID": row["ID"],
                        "Folder": folder,
                        "ROI_Name": roi_name
                    })
        except Exception as e:
            roi_results.append({
                "ID": row["ID"],
                "Folder": folder,
                "ROI_Name": f"Error: {str(e)}"
            })

    # Save results to Excel
    roi_df = pd.DataFrame(roi_results)
    roi_df.to_excel(output_excel, index=False)
    print(f"ROI list saved: {output_excel}")

# Usage of function
extract_rois_from_rtstruct(r"C:\Users\VetteSPM\OneDrive - UMCG\Documents\dicom_summary.xlsx", r"C:\Users\VetteSPM\OneDrive - UMCG\Documents\rtstruct_rois.xlsx")


In [ ]:
# 3. Save CT and RTSTRUCT as nifti - functions

def save_preview_pngs(ct_volume, mask_volume, patient_id, out_dir, roi_name):
    """Save axial and coronal middle slices with overlay."""
    
    out_dir = os.path.join(out_dir, "figures")
    
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    
    spacing = ct_volume.GetSpacing()
    
    ct_volume = sitk.GetArrayFromImage(ct_volume)
    mask_volume = sitk.GetArrayFromImage(mask_volume)
    
    com = ndimage.center_of_mass(mask_volume)
    if np.isnan(com[0]):
        print(f"⚠️ No segmentation found for {patient_id}, skipping preview.")
        return
    
    z_idx = int(com[0])
    y_idx = int(com[1])

    # Axial slice
    plt.figure(figsize=(6, 6))
    plt.imshow(ct_volume[z_idx], cmap="gray")
    plt.imshow(np.ma.masked_where(mask_volume[z_idx] == 0, mask_volume[z_idx]), cmap="jet", alpha=0.5)
    plt.axis("off")
    plt.savefig(os.path.join(out_dir, f"axial_{roi_name}.png"), bbox_inches="tight")
    plt.close()

    # Coronal slice
    plt.figure(figsize=(6, 6))
    plt.imshow(ct_volume[:, y_idx, :][::-1, :], cmap="gray", aspect=spacing[2] / spacing[0])
    plt.imshow(np.ma.masked_where(mask_volume[:, y_idx, :][::-1, :] == 0, mask_volume[:, y_idx, :][::-1, :]), cmap="jet", alpha=0.5, aspect=spacing[2] / spacing[0])
    plt.axis("off")
    plt.savefig(os.path.join(out_dir, f"coronal_{roi_name}.png"), bbox_inches="tight")
    plt.close()
    
    return mask_volume
        
        
def loadCTasSITK(pathCT):
    # Get sorted DICOM filenames
    reader = sitk.ImageSeriesReader()
    series_IDs = reader.GetGDCMSeriesIDs(pathCT)
    if not series_IDs:
        raise RuntimeError(f"No DICOM series found in folder {pathCT}")
    series_file_names = reader.GetGDCMSeriesFileNames(pathCT, series_IDs[0])
    reader.SetFileNames(series_file_names)
    
    # Load the 3D CT image
    image = reader.Execute()
    
    # Get Pixel spacing
    CT_pixel_spacing = image.GetSpacing()
    
    # Load the first slice as a pydicom object to get metadata (!!!!! LAST SLICE GIVES WRONG ImagePosition INFO !!!!!)
    first_slice_dcm = pydicom.dcmread(series_file_names[0])
    ct_imgpos_patMax = [float(x) for x in first_slice_dcm.ImagePositionPatient]
    
    # HU scaling
    intercept = float(first_slice_dcm.RescaleIntercept)
    slope = float(first_slice_dcm.RescaleSlope)
    img_array = sitk.GetArrayFromImage(image).astype(np.float32)
    hu_array = img_array * slope + intercept
    hu_image = sitk.GetImageFromArray(hu_array)
    hu_image.CopyInformation(image)        
    
    return hu_image, CT_pixel_spacing, ct_imgpos_patMax
        
def extract_all_rois_as_nifti(ct_path, rtstruct_path, patient_id, output_folder):
    ct_sitk, _, _ = loadCTasSITK(ct_path)
    rtstruct = RTStructBuilder.create_from(dicom_series_path=ct_path, rt_struct_path=rtstruct_path)

    all_roi_names = rtstruct.get_roi_names()
    # print(f"[INFO] {patient_id} | Found {len(all_roi_names)} ROIs in RTSTRUCT")

    for roi_name in all_roi_names:
        try:
            mask = rtstruct.get_roi_mask_by_name(roi_name)
            mask = np.transpose(mask, (2, 1, 0)).astype(np.uint8)
            mask = np.array([np.rot90(slice_, k=-1) for slice_ in mask])  
            mask = np.flip(mask, axis=2)

            if mask.shape != sitk.GetArrayFromImage(ct_sitk).shape:
                print(f"[WARNING] Shape mismatch for ROI {roi_name}")
                continue

            mask_img = sitk.GetImageFromArray(mask)
            mask_img.CopyInformation(ct_sitk)

            safe_roi_name = re.sub(r'[\\/:"*?<>|]+', '_', roi_name).replace(' ', '_')
            
            out_path = os.path.join(output_folder, f"ROI_{safe_roi_name}.nii.gz")      # MAKE A SUFFIX WHEN RUNNING FOR 1 PATIENT TO NOT OVERWRITE STRUCTURES
            save_nifti(mask_img, out_path)
            
            # Save previews
            mask_volume = save_preview_pngs(ct_sitk, mask_img, patient_id, output_folder, roi_name)

        except Exception as e:
            print(f"[ERROR] Failed to process ROI '{roi_name}': {e}")

    # Save the CT itself
    save_nifti(ct_sitk, os.path.join(output_folder, "CT.nii.gz"))
    
    return mask_volume
        
def save_nifti(sitk_img, path):
    arr = sitk.GetArrayFromImage(sitk_img)  # (z, y, x)
    affine = np.diag(sitk_img.GetSpacing()[::-1] + (1,))
    nib_img = nib.Nifti1Image(arr, affine)
    nib.save(nib_img, path)        


In [ ]:
# 3. Save CT and RTSTRUCT as nifti - main loop
excel_file = r"C:\Users\VetteSPM\OneDrive - UMCG\Documents\dicom_summary.xlsx"
    
df = pd.read_excel(excel_file)

for patient_id, group in df.groupby("ID"):
    ct_folders = group[group["Modality"] == "CT"]["Folder"].tolist()
    rtstruct_folders = group[group["Modality"] == "RTSTRUCT"]["Folder"].tolist()

    if not ct_folders or not rtstruct_folders:
        print(f"⚠️ Skipping {patient_id}: missing CT or RTSTRUCT")
        continue

    ct_folder = ct_folders[0]  # take first CT folder
    rtstruct_folder = rtstruct_folders[0]  # take first RTSTRUCT folder
    rtstruct_file = [os.path.join(rtstruct_folder, f) for f in os.listdir(rtstruct_folder) if f.endswith(".dcm")][0]
    
    out_dir =  r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed"
    out_dir = os.path.join(out_dir, patient_id)
    
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    
    try:
        hu_image, CT_pixel_spacing, ct_imgpos_patMax = loadCTasSITK(ct_folder)        
        mask_volume = extract_all_rois_as_nifti(ct_folder, rtstruct_file, patient_id, out_dir)
        # print(f"✅ Processed {patient_id}")
        
    except Exception as e:
        print(f"❌ Error with {patient_id}: {e}")

In [2]:
# 4. Cropping, resampling, intensity scaling functions
def load_nifti(path):
    nii = nib.load(path)
    return nii.get_fdata(), nii.affine, nii.header
    
def resample_to_spacing(volume, affine, new_spacing=(2.0, 2.0, 2.0), order=3):
    """
    Resample a 3D volume to new voxel spacing.
    order=3 for CT (cubic), order=0 for masks (nearest).
    """
    # Extract current voxel spacing from affine
    current_spacing = np.abs(affine[:3, :3]).diagonal()
    
    zoom_factors = current_spacing / np.array(new_spacing)
    resampled = zoom(volume, zoom_factors, order=order)
    return resampled

def crop_fixed_size(volume, center, size):
    """
    Crop a fixed-size 3D volume around a given center.
    size = (x, y, z) in voxels
    """
    x, y, z = volume.shape
    sx, sy, sz = size
    
    cx, cy, cz = center
    
    # Compute bounds
    x_min = max(cx - sx // 2, 0)
    y_min = max(cy - sy // 2, 0)
    z_min = max(cz - sz // 2, 0)
    
    x_max = min(x_min + sx, x)
    y_max = min(y_min + sy, y)
    z_max = min(z_min + sz, z)
    
    cropped = volume[x_min:x_max, y_min:y_max, z_min:z_max]
    
    # If crop is smaller than desired size, pad with zeros
    pad_x = sx - cropped.shape[0]
    pad_y = sy - cropped.shape[1]
    pad_z = sz - cropped.shape[2]
    
    if pad_x > 0 or pad_y > 0 or pad_z > 0:
        cropped = np.pad(
            cropped,
            ((0, pad_x), (0, pad_y), (0, pad_z)),
            mode="constant",
            constant_values=0
        )
    
    return cropped    

def normalize_ct(ct, min_hu=-1000, max_hu=400):
    """
    Clip CT to [min_hu, max_hu] and scale to [0, 1]
    """
    ct_clipped = np.clip(ct, min_hu, max_hu)
    ct_normalized = (ct_clipped - min_hu) / (max_hu - min_hu)
    return ct_normalized.astype(np.float32)

def process_patient(patient_folder, output_folder):
    ct_path = os.path.join(patient_folder, "CT.nii.gz")
    left_path = os.path.join(patient_folder, "ROI_Lung-Left.nii.gz")
    right_path = os.path.join(patient_folder, "ROI_Lung-Right.nii.gz")
    total_path = os.path.join(patient_folder, "ROI_Lungs-Total.nii.gz")
    
    # Load CT
    ct, affine, header = load_nifti(ct_path)
    
    # Load masks and combine
    if os.path.exists(total_path):
        # Use total mask if it exists
        mask, _, _ = load_nifti(total_path)
        lung_mask = mask > 0
    else:
        # Otherwise combine left and right
        left, _, _ = load_nifti(left_path)
        right, _, _ = load_nifti(right_path)
        lung_mask = (left > 0) | (right > 0)

    # Resample both CT and mask
    ct_resampled = resample_to_spacing(ct, affine, new_spacing=(2, 2, 2), order=3)
    mask_resampled = resample_to_spacing(lung_mask.astype(np.uint8), affine, new_spacing=(2, 2, 2), order=0)

    # Find lung center
    coords = np.array(np.where(mask_resampled))
    center = coords.mean(axis=1).astype(int)  # (x, y, z)

    # Crop to fixed size
    target_size = (180, 180, 150)
    ct_cropped = crop_fixed_size(ct_resampled, center, target_size)
    mask_cropped = crop_fixed_size(mask_resampled, center, target_size)

    ct_normalized = normalize_ct(ct_cropped, min_hu=ct_cropped.min(), max_hu=ct_cropped.min() + 1424)
    
    ct_reoriented = np.transpose(ct_normalized, (1, 2, 0))   # (z,x,y) → (x,y,z)
    mask_reoriented = np.transpose(mask_cropped, (1, 2, 0))
    
    # Save as numpy arrays
    os.makedirs(output_folder, exist_ok=True)
    np.save(os.path.join(output_folder, "CT_resampled_cropped.npy"), ct_reoriented.astype(np.float32))
    np.save(os.path.join(output_folder, "LungMask_resampled_cropped.npy"), mask_reoriented.astype(np.uint8))

    print("Saved:", output_folder)
  


In [3]:
# 4. Cropping, resampling, intensity scaling main loop
data_folder = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed"
output_root = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2"

# List all subfolders (patients)
patient_folders = [os.path.join(data_folder, d) for d in os.listdir(data_folder)
                   if os.path.isdir(os.path.join(data_folder, d))]

for patient_folder in patient_folders:
    patient_id = os.path.basename(patient_folder)
    output_folder = os.path.join(output_root, patient_id)
    os.makedirs(output_folder, exist_ok=True)
    
    print(f"Processing patient {patient_id}...")

    try:
        # Call your existing processing function
        # This function should do: load CT/mask, resample, crop, intensity clip, save .npy
        process_patient(patient_folder, output_folder)

    except Exception as e:
        print(f"Failed to process {patient_id}: {e}")



Processing patient LUNG1-001...
Saved: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-001
Processing patient LUNG1-002...
Saved: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-002
Processing patient LUNG1-003...
Saved: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-003
Processing patient LUNG1-004...
Saved: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-004
Processing patient LUNG1-005...
Saved: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-005
Processing patient LUNG1-006...
Failed to process LUNG1-006: No such file or no access: '//zkh/appdata/RTDicom/Projectline_HNC_modelling/Public_datasets/TCIA_lung_processed/LUNG1-006/CT.nii.gz'
Processing patient LUNG1-007...
Saved: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-007
Pr

In [4]:
# 5. Delete empty folders (patientIDs that failed)

# Top-level folder containing patient subfolders
data_folder = output_root

for root, dirs, files in os.walk(data_folder, topdown=False):
    for d in dirs:
        folder_path = os.path.join(root, d)
        # Check if folder is empty (no files and no subfolders)
        if not os.listdir(folder_path):
            print(f"Deleting empty folder: {folder_path}")
            shutil.rmtree(folder_path)

# Result: 17 patients failed (no lung mask or mismatch UID CT and RTSTRUCT)

Deleting empty folder: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-006
Deleting empty folder: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-035
Deleting empty folder: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-050
Deleting empty folder: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-067
Deleting empty folder: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-115
Deleting empty folder: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-119
Deleting empty folder: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-121
Deleting empty folder: \\zkh\appdata\RTDicom\Projectline_HNC_modelling\Public_datasets\TCIA_lung_processed2\LUNG1-128
Deleting empty folder: \\zkh\appdata\RTDicom\Projectline